In [3]:
from pathlib import Path
from fontTools.ttLib import TTFont

root_dir = r"C:\Users\Admin\Desktop\projects\write2font\ttf"
ttffiles = Path(root_dir).rglob("*.ttf")

print("🔄 TTF 파일에서 지원하는 글자 목록 추출 시작...")

for ttffile in ttffiles:
    txtfile = ttffile.with_suffix(".txt")

    # 이미 txt 파일이 있으면 건너뛰기
    if txtfile.is_file():
        print(f"⏩ 패스: {txtfile.name} (이미 존재함)")
        continue

    try:
        # fontTools를 사용해 강제로 글자 목록(cmap) 추출
        font = TTFont(str(ttffile))
        cmap = font['cmap'].getBestCmap()

        if cmap:
            chars = [chr(k) for k in cmap.keys()]
            with open(txtfile, "w", encoding="utf-8") as f:
                f.write("".join(chars))
            print(f"✅ 성공: {txtfile.name} 생성 완료!")
        else:
            print(f"⚠️ 경고: {ttffile.name} 내부에 글자 데이터가 없습니다.")

    except Exception as e:
        print(f"🚨 실패: {ttffile.name} - 상세 에러: {e}")

print("✨ 모든 작업이 끝났습니다!")

🔄 TTF 파일에서 지원하는 글자 목록 추출 시작...
⏩ 패스: gaeun_handwriting.txt (이미 존재함)
⏩ 패스: OwnglyphGeumhyang.txt (이미 존재함)
⏩ 패스: SeoyeonHandwriting.txt (이미 존재함)
✅ 성공: 온글잎 재건사.txt 생성 완료!
✨ 모든 작업이 끝났습니다!


In [ ]:
import os
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from tqdm import tqdm

# 1. 경로 설정
ttf_dir = r"C:\Users\Admin\Desktop\projects\write2font\ttf"
output_base_dir = r"C:\Users\Admin\Desktop\projects\write2font\png"

# 2. 렌더링 설정
# image_size = 128 (이제 고정된 캔버스 크기는 사용하지 않으므로 제거합니다)
font_size = 100 # 글자의 해상도(디테일)를 결정하므로 100 유지

Path(output_base_dir).mkdir(parents=True, exist_ok=True)
ttf_files = list(Path(ttf_dir).rglob("*.ttf"))

print(f"🔄 총 {len(ttf_files)}개의 TTF 파일을 '여백 없이(Tight Crop)' 변환합니다...\n")

# ⚠️ 파일명으로 쓰면 에러가 나는 위험한 특수문자 목록
forbidden_chars = '\\/:*?"<>|.'

for ttf_path in ttf_files:
    font_name = ttf_path.stem
    txt_path = ttf_path.with_suffix(".txt")

    save_dir = Path(output_base_dir) / font_name
    save_dir.mkdir(parents=True, exist_ok=True)

    if not txt_path.exists():
        print(f"⚠️ {font_name}.txt 파일이 없습니다. 건너뜁니다.")
        continue

    with open(txt_path, "r", encoding="utf-8") as f:
        chars = f.read().strip()

    try:
        font = ImageFont.truetype(str(ttf_path), font_size)
    except Exception as e:
        print(f"🚨 폰트 로드 실패 ({font_name}): {e}")
        continue

    print(f"👉 [{font_name}] 폰트 렌더링 중... (총 {len(chars)}자)")

    for char in tqdm(chars):
        # 🛡️ 에러 방지: 특수문자나 제어문자는 유니코드 헥사값(예: U+002F)으로 이름 변경
        if char in forbidden_chars or ord(char) < 32:
            safe_name = f"U+{ord(char):04X}"
        else:
            safe_name = char

        # --- [수정된 핵심 로직: 타이트 크롭 렌더링] ---
        # 1. 글자의 타이트한 경계 상자(Bounding Box) 추출
        bbox = font.getbbox(char)
        
        # 보이지 않는 공백 문자(Space 등) 처리
        if bbox is None:
            continue
            
        # bbox: (left, top, right, bottom)
        w = bbox[2] - bbox[0]
        h = bbox[3] - bbox[1]
        
        # 글자가 렌더링되지 않는 빈 칸 오류 방지
        if w <= 0 or h <= 0:
            continue

        # 2. 딱 알맹이 크기(w x h)만큼의 빈 캔버스 생성 (배경: 0=검은색)
        img = Image.new('L', (w, h), color=0)
        draw = ImageDraw.Draw(img)

        # 3. 폰트 고유의 여백(Offset)을 무시하고 (0,0)에 딱 달라붙게 음수 좌표 부여
        x = -bbox[0]
        y = -bbox[1]

        # 4. 여백 없이 렌더링 (글자: 255=흰색)
        draw.text((x, y), char, fill=255, font=font)
        # -----------------------------------------------

        try:
            # 안전하게 변환된 이름으로 저장 (예: 가.png 또는 U+002F.png)
            img.save(save_dir / f"{safe_name}.png")
        except Exception as e:
            pass # 혹시 모를 다른 OS단 파일명 에러는 가볍게 무시하고 다음 글자로 진행

print("\n✨ 모든 TTF 폰트의 [타이트 크롭 PNG] 렌더링이 완료되었습니다!")

🔄 총 4개의 TTF 파일을 '여백 없이(Tight Crop)' 변환합니다...

👉 [gaeun_handwriting] 폰트 렌더링 중... (총 11266자)


100%|██████████████████████████████████████████████████████████████████████████| 11266/11266 [00:01<00:00, 6058.18it/s]


👉 [OwnglyphGeumhyang] 폰트 렌더링 중... (총 11413자)


 53%|███████████████████████████████████████▉                                   | 6082/11413 [00:04<00:03, 1446.63it/s]